In [1]:
# #生成数据
# import numpy as np
# import networkx as nx

# n = 400
# p = 0.5

# # 生成随机图
# G = nx.gnp_random_graph(n, p)

# # 计算每个节点的邻居数量
# num_nb = np.array([len(list(G.neighbors(node))) for node in G.nodes()])

# # 将邻接矩阵进行归一化
# adj_matrix = nx.to_numpy_array(G)
# G = adj_matrix / np.sum(adj_matrix, axis=1, keepdims=True)

# print('G:', G)
# print('num_nb:', num_nb)

import numpy as np
import networkx as nx

#np.set_printoptions(threshold = np.inf)
#np.set_printoptions(suppress = True)

n = 400
degree_max = 5

# 生成度序列
degree_sequence = [np.random.randint(1, degree_max) for _ in range(n)]
# degree_sequence = np.ceil(degree_sequence)
# degree_sequence.astype(int)
while sum(degree_sequence) % 2 != 0:  # 确保度序列的总和是偶数
    degree_sequence = [np.random.randint(1, degree_max) for _ in range(n)]
#     degree_sequence = np.ceil(degree_sequence)
#     degree_sequence.astype(int)

G = nx.configuration_model(degree_sequence)

# 将多重边合并为单一边
G = nx.Graph(G)

# 生成邻接矩阵并计算邻居数量
adj_matrix = nx.to_numpy_array(G)
E = adj_matrix
num_nb = np.sum(E, axis=1)

# 计算G
G = E / np.sum(E, axis=1, keepdims=True)


print('G:', G)
print('E:', E)
print('num_nb:', num_nb)



G: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
E: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
num_nb: [4. 1. 4. 1. 4. 4. 1. 1. 4. 2. 2. 3. 3. 4. 2. 3. 3. 3. 4. 2. 4. 4. 4. 2.
 1. 3. 3. 4. 4. 3. 2. 1. 1. 4. 4. 4. 4. 1. 1. 2. 1. 3. 1. 4. 4. 4. 1. 1.
 3. 2. 3. 1. 2. 2. 4. 3. 4. 3. 1. 4. 4. 4. 1. 2. 4. 4. 2. 3. 4. 2. 1. 3.
 4. 2. 2. 1. 3. 4. 3. 3. 2. 3. 3. 4. 3. 4. 2. 1. 4. 3. 4. 3. 3. 2. 3. 3.
 3. 3. 2. 1. 4. 4. 3. 1. 2. 2. 3. 1. 4. 4. 3. 1. 3. 4. 2. 4. 3. 1. 4. 2.
 2. 1. 1. 2. 4. 2. 4. 4. 4. 1. 4. 3. 1. 2. 1. 2. 2. 3. 3. 4. 4. 2. 4. 4.
 3. 2. 4. 3. 2. 3. 3. 2. 3. 3. 2. 2. 4. 2. 3. 1. 1. 3. 4. 4. 2. 3. 3. 1.
 4. 3. 2. 3. 4. 2. 4. 1. 2. 1. 1. 4. 1. 1. 1. 3. 2. 1. 4. 4. 4. 4. 1. 3.
 1. 4. 4. 4. 2. 1. 3. 4. 4. 4. 4. 2. 2. 4. 1. 1. 3. 4. 2. 4. 1. 3. 2. 4.
 4. 1. 1. 1. 4. 1

In [2]:
import numpy as np

# 对长度为n的正态分布随机数向量进行标准化
X = np.random.normal(size=n)
X = (X - np.mean(X)) / np.std(X, ddof=1)

# 生成长度为n的正态分布随机误差项epsilon
epsilon = np.random.normal(size=n)

# define the simulated outcomes and the exposure mapping function:
def get_Y(Z):
    return np.linalg.solve(-0.8*G+ np.diag([1]*n), -np.ones(n) + G.dot(Z) + Z + X + epsilon)

#这里可以换用多种仿真策略

def get_T(Z):
    return (E.dot(Z) >= np.floor(num_nb/2)).astype(int)
#注意大于等于号



In [3]:
from scipy.stats import binom
import numpy as np

r1 = 0.5
# 计算 pscore1 和 pscore0
pscore1 = binom.cdf(np.floor(num_nb/2), num_nb, r1)
pscore0 = 1 - pscore1

print('pscore1:', pscore1)
print('pscore0:', pscore0)

# 构建 A 矩阵
A = np.dot(E, E) > 0

# 计算 A_p 矩阵
temp = np.linalg.eig(A)
vectors = temp[1]
values = temp[0]
positive_values = values * (values > 0)
A_p = np.dot(np.dot(vectors, np.diag(positive_values)), np.linalg.inv(vectors))


pscore1: [0.6875 0.5    0.6875 0.5    0.6875 0.6875 0.5    0.5    0.6875 0.75
 0.75   0.5    0.5    0.6875 0.75   0.5    0.5    0.5    0.6875 0.75
 0.6875 0.6875 0.6875 0.75   0.5    0.5    0.5    0.6875 0.6875 0.5
 0.75   0.5    0.5    0.6875 0.6875 0.6875 0.6875 0.5    0.5    0.75
 0.5    0.5    0.5    0.6875 0.6875 0.6875 0.5    0.5    0.5    0.75
 0.5    0.5    0.75   0.75   0.6875 0.5    0.6875 0.5    0.5    0.6875
 0.6875 0.6875 0.5    0.75   0.6875 0.6875 0.75   0.5    0.6875 0.75
 0.5    0.5    0.6875 0.75   0.75   0.5    0.5    0.6875 0.5    0.5
 0.75   0.5    0.5    0.6875 0.5    0.6875 0.75   0.5    0.6875 0.5
 0.6875 0.5    0.5    0.75   0.5    0.5    0.5    0.5    0.75   0.5
 0.6875 0.6875 0.5    0.5    0.75   0.75   0.5    0.5    0.6875 0.6875
 0.5    0.5    0.5    0.6875 0.75   0.6875 0.5    0.5    0.6875 0.75
 0.75   0.5    0.5    0.75   0.6875 0.75   0.6875 0.6875 0.6875 0.5
 0.6875 0.5    0.5    0.75   0.5    0.75   0.75   0.5    0.5    0.6875
 0.6875 0.75   0.6875 0.

In [4]:
import numpy as np

def get_X(X, Z, G):
    return np.column_stack((Z, G.dot(Z), X, G.dot(X)))

def get_orth_coef(X, G, num_rep=10000):
    mom_mat = np.zeros((num_rep, 5))
    for i in range(num_rep):
        Z = np.random.binomial(1, r1, n)
        X_aug = get_X(X, Z, G)
        T_vec = get_T(Z)
        w = T_vec/pscore1 - (1 - T_vec)/pscore0
        mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w)]

    mom_mean = np.mean(mom_mat, axis=0)
    return mom_mean[1:]/mom_mean[0]

orth_coef = get_orth_coef(X, G)

print('orth_coef:', orth_coef)

orth_coef: [0.09552559 0.17003183 0.00456352 0.00594717]


In [5]:
# np.ceil([0.1,0.2])

# # np.random.randint(0, 10) for _ in range(n)
# degree_sequence = [np.random.uniform(0, 10) for _ in range(n)]
# # degree_sequence = np.ceil(degree_sequence)
# degree_sequence.astype(int)
# print(degree_sequence)

# 初始化一个空列表来存储计算结果
tau = []

# 进行10000次循环
for _ in range(1000):
    Z = np.random.binomial(1, r1, n)  # 生成服从二项分布的随机数Z
    Y = get_Y(Z)  # 根据Z计算Y
    T_vec = get_T(Z)  # 根据Z计算T_vec,分为exposure mapping中treated negighborhood是否过半的01两类
    
    estimate = Y*T_vec/pscore1 - Y*(1 - T_vec)/pscore0  # 根据Y、T_vec和pscore计算D
    tau_unadj_ht = np.mean(estimate)  # 计算未调整的tau值
    tau.append(tau_unadj_ht)  # 将tau_unadj_ht添加到列表中

# 计算所有tau_unadj_ht的均值
tau_mean = np.mean(tau)


#这是NAIVE的估计：unadjusted HT estiamtor
# 打印均值
print(tau_mean)


1.2280706633193397


In [6]:
import scipy
from scipy.stats import binom, norm
# sim_res_1<- map_dfr(1:1000, ~{
#   Z <- rbinom(n, size = 1, prob = r1); Y <- get_Y(Z); X_aug <- 
#     (X,Z,G)
#   T_vec <- get_T(Z); D <- Y*T_vec/pscore1-Y*(1-T_vec)/pscore0; 
#   w <- T_vec/pscore1-(1-T_vec)/pscore0
    
#   tau_unadj_ht <- mean(D); 
  
#   var_est_unadj_1 <- t(D-tau_unadj_ht)%*%A%*%(D-tau_unadj_ht)/n^2 %>% as.vector();
#   var_est_unadj_2 <- t(D-tau_unadj_ht)%*%A_p%*%(D-tau_unadj_ht)/n^2 %>% as.vector()
    
#   is_cover_unadj_1 <- abs(tau_unadj_ht-tau)<qnorm(0.975)*sqrt(var_est_unadj_1)
#   is_cover_unadj_2 <- abs(tau_unadj_ht-tau)<qnorm(0.975)*sqrt(var_est_unadj_2)
  
#   return(tibble(tau_unadj_ht, var_est_unadj_1, var_est_unadj_2, is_cover_unadj_1, is_cover_unadj_2 ))
# })

import numpy as np
import pandas as pd

sim_res_1 = []

for _ in range(1000):
    Z = np.random.binomial(1, r1, n)  # 生成服从二项分布的随机数Z
#     print('Z', Z)
    Y = get_Y(Z)  # 根据Z计算Y
#     print('X:',X)
#     print('Z:', Z)
#     print('G:', G)

    X_aug = get_X(X, Z, G)  # 生成X_aug（应该是X, Z, G的组合）
    T_vec = get_T(Z)  # 根据Z计算T_vec
    D = Y*T_vec/pscore1 - Y*(1 - T_vec)/pscore0  # 根据Y、T_vec和pscore计算D
    w = T_vec/pscore1 - (1 - T_vec)/pscore0
    tau_unadj_ht = np.mean(D)  # 计算未调整的tau值
    
    var_est_unadj_1 = np.dot(np.dot((D-tau_unadj_ht).T, A), (D-tau_unadj_ht)) / n**2
    var_est_unadj_2 = np.dot(np.dot((D-tau_unadj_ht).T, A_p), (D-tau_unadj_ht)) / n**2
    
    is_cover_unadj_1 = np.abs(tau_unadj_ht - tau) < scipy.stats.norm.ppf(0.975) * np.sqrt(var_est_unadj_1)
    is_cover_unadj_2 = np.abs(tau_unadj_ht - tau) < norm.ppf(0.975) * np.sqrt(var_est_unadj_2)
    
    sim_res_1.append([tau_unadj_ht, var_est_unadj_1, var_est_unadj_2, is_cover_unadj_1, is_cover_unadj_2])

sim_res_1_df = pd.DataFrame(sim_res_1, columns=['tau_unadj_ht', 'var_est_unadj_1', 'var_est_unadj_2', 'is_cover_unadj_1', 'is_cover_unadj_2'])

print('sim_res_1_df:', sim_res_1_df)






sim_res_1_df:      tau_unadj_ht  var_est_unadj_1     var_est_unadj_2  \
0        0.504620         0.094363  0.136788-0.000000j   
1        1.219288         0.142663  0.184813+0.000000j   
2        0.726396         0.137044  0.174094-0.000000j   
3        1.247982         0.151624  0.173311-0.000000j   
4        1.681950         0.206474  0.240937+0.000000j   
..            ...              ...                 ...   
995      0.593189         0.124243  0.158116-0.000000j   
996      1.124677         0.131879  0.174079-0.000000j   
997      1.282422         0.151000  0.186376+0.000000j   
998      1.240469         0.144132  0.183752-0.000000j   
999      1.581606         0.137296  0.169200-0.000000j   

                                      is_cover_unadj_1  \
0    [False, False, False, False, False, True, Fals...   
1    [True, True, True, True, True, True, True, Tru...   
2    [True, False, True, True, True, True, True, Tr...   
3    [True, True, True, True, True, True, True, Tru...   

In [37]:
import numpy as np
import pandas as pd
from scipy.stats import binom, norm
from sklearn.preprocessing import scale




# sim_res_2<- map_dfr(1:1000, ~{
#   Z <- rbinom(n, size = 1, prob = r1); Y <- get_Y(Z); X_aug <- 
#     (X,Z,G)
#   T_vec <- get_T(Z); D <- Y*T_vec/pscore1-Y*(1-T_vec)/pscore0; 
#   w <- T_vec/pscore1-(1-T_vec)/pscore0
#   tau_unadj_ht <- mean(D); 

#   X_db <- X_aug-(w)%*%t(orth_coef)
#   D_2 <- scale(X_db*w, scale = FALS
#               ) 
#   hbeta_1 <- solve(t(D_2)%*%A%*%(D_2),t(D_2)%*%A%*%(D-tau_unadj_ht))
#   hbeta_2 <- solve(t(D_2)%*%A_p%*%(D_2),t(D_2)%*%A_p%*%(D-tau_unadj_ht))
  
#   tau_adj_aug1 <- mean((Y-X_db%*%hbeta_1)*w)
#   tau_adj_aug2 <- mean((Y-X_db%*%hbeta_2)*w)
  
#   var_est_adj_aug1 <- t(D-D_2%*%hbeta_1-tau_adj_aug1)%*%A%*%(D-D_2%*%hbeta_1-tau_adj_aug1)/n^2 %>% as.vector()
#   var_est_adj_aug2 <- t(D-D_2%*%hbeta_2-tau_adj_aug2)%*%A_p%*%(D-D_2%*%hbeta_2-tau_adj_aug2)/n^2 %>% as.vector()
  
#   is_cover_adj_aug_1 <- abs(tau_adj_aug1-tau)<qnorm(0.975)*sqrt(var_est_adj_aug1)
#   is_cover_adj_aug_2 <- abs(tau_adj_aug2-tau)<qnorm(0.975)*sqrt(var_est_adj_aug2)
  
#   return(tibble(tau_adj_aug1,tau_adj_aug2, var_est_adj_aug1, var_est_adj_aug2, is_cover_adj_aug_1, is_cover_adj_aug_2))
# })

# #这是第二种做回归调整的情况，是HT estimator
sim_res_2 = []

for _ in range(1000):
    Z = np.random.binomial(1, r1,n)  # rbinom 函数的Python等价代码
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
    X_aug = get_X(X, Z, G)  # 合并X, Z, G的数据，得到X_aug

    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
    w = T_vec / pscore1 - (1 - T_vec) / pscore0
    tau_unadj_ht = np.mean(D)
   
    print('X_aug:', X_aug.shape)
    print('w:', w.shape)
    print('orth_coef.T:', orth_coef.T.shape)
    
#     X_db = X_aug - np.dot(w, orth_coef)  # 公式计算; 看似是point-wise estimation


    X_db = X_aug - np.dot(w.reshape(-1, 1), orth_coef.reshape(1, -1))

#     D_2 = (X_db * w)  # scale 函数的Python等价代码
    D_2 = scale(X_db * np.array(w).reshape(-1,1), axis=0, with_std=False)

    hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A), (D - tau_unadj_ht)))
    hbeta_2 = np.linalg.solve(np.dot(np.dot(D_2.T, A_p), D_2), np.dot(np.dot(D_2.T, A_p), (D - tau_unadj_ht)))

    tau_adj_aug1 = np.mean((Y - np.dot(X_db, hbeta_1)) * w)
    tau_adj_aug2 = np.mean((Y - np.dot(X_db, hbeta_2)) * w)

    var_est_adj_aug1 = np.dot(np.dot((D - np.dot(D_2, hbeta_1) - tau_adj_aug1).T, A), (D - np.dot(D_2, hbeta_1) - tau_adj_aug1)) / n**2
    var_est_adj_aug2 = np.dot(np.dot((D - np.dot(D_2, hbeta_2) - tau_adj_aug2).T, A_p), (D - np.dot(D_2, hbeta_2) - tau_adj_aug2)) / n**2

    is_cover_adj_aug_1 = abs(tau_adj_aug1 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug1)
    is_cover_adj_aug_2 = abs(tau_adj_aug2 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug2)
    print('is_cover_adj_aug_1:', is_cover_adj_aug_1.shape)
    
    sim_res_2.append([tau_adj_aug1, tau_adj_aug2, var_est_adj_aug1, var_est_adj_aug2, is_cover_adj_aug_1, is_cover_adj_aug_2])

sim_res_2 = pd.DataFrame(sim_res_2, columns=['tau_adj_aug1', 'tau_adj_aug2', 'var_est_adj_aug1', 'var_est_adj_aug2', 'is_cover_adj_aug_1', 'is_cover_adj_aug_2'])

print(sim_res_2)

X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: 

is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.

is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.

is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.

X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: 

X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: 

is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.

is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.T: (4,)
is_cover_adj_aug_1: (1000,)
X_aug: (400, 4)
w: (400,)
orth_coef.

In [39]:
import numpy as np
import pandas as pd
from scipy.stats import binom, norm
import statsmodels.api as sm



# sim_res_3<- map_dfr(1:1000, ~{
#   Z <- rbinom(n, size = 1, prob = r1); Y <- get_Y(Z); X_aug <- 
#     (X,Z,G)
#   T_vec <- get_T(Z); D <- Y*T_vec/pscore1-Y*(1-T_vec)/pscore0; 
#   w <- T_vec/pscore1-(1-T_vec)/pscore0
  
#   lm_haj <- lm(Y~1+T_vec+X+T_vec:X, w = T_vec/pscore1+(1-T_vec)/pscore0)
#   e_haj <- lm_haj %>% resid(); tau_adj_haj <- lm_haj %>% coef() %>% .[2]
#   C_haj <- cbind(1,T_vec,X,T_vec*X)
#   var_est_adj_haj <- solve(t(C_haj)%*%diag(w)%*%C_haj)%*%(t(C_haj)%*%diag(w)%*%diag(e_haj)%*%A_p%*%diag(e_haj)%*%diag(w)%*%(C_haj))%*%solve(t(C_haj)%*%diag(w)%*%C_haj) %>% .[2,2]
#   is_cover_adj_haj <- abs(tau_adj_haj-tau)<qnorm(0.975)*sqrt(var_est_adj_haj)
  
#   return(tibble(tau_adj_haj, var_est_adj_haj , is_cover_adj_haj))
# })


sim_res_3 = []


for _ in range(1000):
    Z = np.random.binomial(1, r1, n)  # rbinom 函数的Python等价代码
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
    X_aug = get_X(X, Z, G)  # 合并X, Z, G的数据，得到X_aug

    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
    w = T_vec / pscore1 - (1 - T_vec) / pscore0
    print('w.shape:', w.shape)
    #w = [1]* n
    w = np.array(w)
    print('w.shape:', w.shape)
    print('Y:', Y)
    print('w:', w)
    print('covariate:', np.column_stack((T_vec, X, T_vec*X)))
    lm_haj = sm.WLS(Y, sm.add_constant(np.column_stack((T_vec, X, T_vec*X))), weights=w, missing='drop').fit()  # 使用加权最小二乘法拟合模型
    

    
    e_haj = lm_haj.resid
    tau_adj_haj = lm_haj.params[2]  # 获取模型参数
    print('T_vec:',T_vec.shape)
    C_haj = np.column_stack((np.ones(len(X_aug)), T_vec, X_aug, np.array(T_vec).reshape(-1,1)*X_aug))
#     var_est_adj_haj = np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.linalg.inv(np.dot(np.dot(C_haj.T, np.diag(w)), C_haj)),
#                                                       C_haj.T), np.diag(w)), np.diag(e_haj)), A_p), np.diag(e_haj)), np.diag(w)), C_haj))
 

    temp_term = np.linalg.inv(C_haj.T @ np.diag(w) @ C_haj) @ (C_haj.T @ np.diag(w) @ np.diag(e_haj) @ A_p @ np.diag(e_haj) @ np.diag(w) @ C_haj) @ np.linalg.inv(C_haj.T @ np.diag(w) @ C_haj)
    var_est_adj_haj = temp_term[1, 1]

    
    
    is_cover_adj_haj = abs(tau_adj_haj - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_haj)

    sim_res_3.append([tau_adj_haj, var_est_adj_haj, is_cover_adj_haj])

sim_res_3 = pd.DataFrame(sim_res_3, columns=['tau_adj_haj', 'var_est_adj_haj', 'is_cover_adj_haj'])

print('sim_res_3:' , sim_res_3)


w.shape: (400,)
w.shape: (400,)
Y: [ -4.0533186    2.1625754    1.29432482   0.13818605  -1.83314178
  -3.65652185   0.9403201   -4.89713625  -1.28109128  -3.84694514
  -4.75725125   0.47012742  -3.52495238  -0.11501725   1.8272884
   2.76404698   1.70570807   1.19672436  -0.61644481   4.24629322
  -1.55117048   3.1755713   -4.03302359  -7.57219609  -3.35068041
   0.40371149  -3.01258966   0.78562594   0.46008061   0.74843341
   2.26856094  -0.61652069  -0.59088157   0.49533438  -3.26764308
  -1.67227873   0.06477148   1.75359977   4.06237508  -5.08652441
  -1.20200062  -1.40734734  -8.82551696   0.16057052  -1.15724804
   2.18558834  -5.64834987  -2.38958658   0.38447444   3.98670491
  -1.7554662    1.6581665    3.05942917  -0.34742992  -0.87487294
   0.31796483  -1.49800404   2.99826889   0.91833159  -0.52645842
  -1.31306055   0.90428438   2.89305276   0.08627339  -5.62763172
   0.08311342  -3.7847988   -2.03396241  -3.05449889   1.82816898
  -2.07511126  -1.13470476  -4.24693731  -

/Users/zhangzhiheng/opt/anaconda3/lib/python3.8/site-packages/statsmodels/regression/linear_model.py:731: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(self.weights)[:, None] * x
/Users/zhangzhiheng/opt/anaconda3/lib/python3.8/site-packages/statsmodels/regression/linear_model.py:729: RuntimeWarning: invalid value encountered in sqrt
  return x * np.sqrt(self.weights)


LinAlgError: SVD did not converge

In [ ]:
a = [1]*4
a
